<a href="https://colab.research.google.com/github/SharikaTasnim23/Thesis/blob/main/Final_Individual_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# Cell 1: Environment and Libraries Setup
!pip install librosa pandas numpy tensorflow scikit-learn

import os
import re
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
import random
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, accuracy_score, f1_score

# রেজাল্টের পুনরাবৃত্তি ঠিক রাখতে সিড সেট করা
random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)
# জিপিইউ অপারেশনকে যতটা সম্ভব ডিটারমিনিস্টিক করা
tf.config.experimental.enable_op_determinism()

print("TensorFlow Version:", tf.__version__)
print("Cell 1: Environment is ready!")

TensorFlow Version: 2.20.0
Cell 1: Environment is ready!


In [ ]:
# Cell 2: Global Constants and Paths
from google.colab import drive
drive.mount('/content/drive')

CSV_PATH = "/content/drive/MyDrive/Wolfset_Dataset/Wolfset_Audio_files/Wolfset_Index.csv"
AUDIO_DIR = "/content/drive/MyDrive/Wolfset_Dataset/Wolfset_Audio_files"

# অডিও প্রসেসিং প্যারামিটার
SR = 22050                 # মোটর ও শিপিং নয়েজের জন্য আদর্শ স্যাম্পলিং রেট
SEGMENT_SEC = 10            # ৩ সেকেন্ডের উইন্ডো সাইজ
SEGMENT_SAMPLES = SR * SEGMENT_SEC

# ফিচার এক্সট্র্যাকশন প্যারামিটার (টোটাল ১৭৪টি ফিচার আসবে)
N_FFT = 2048               # ফ্রিকোয়েন্সি রেজোলিউশন বাড়ানোর জন্য বড় উইন্ডো
HOP_LENGTH = 512           # ৭৫% ইন্টারনাল ওভারল্যাপ
N_MELS = 128               # ১২৮টি মেল ব্যান্ড
N_MFCC = 40                # ৪০টি এমএফসিসি কোফিসিয়েন্ট
N_BANDS = 6                # ৬টি স্পেকট্রাল কনট্রাস্ট ব্যান্ড

# টোটাল ফিচার = (১২৮ মেল + ৪০ এমএফসিসি + ৬ কনট্রাস্ট) = ১৭৪ (Mean) + ১৭৪ (Std) = ৩৪৮
print(f"Target Feature Vector Size per segment: { (N_MELS + N_MFCC + N_BANDS) * 2 } (348, 1)")

Mounted at /content/drive
Target Feature Vector Size per segment: 348 (348, 1)


In [ ]:
# Cell 3: Structural File Name Parsing using Regex Pattern
FILENAME_PATTERN = r"^[AE](\d{5})T\d{2}N\d{2}(?:x\d+)?"

# ৫টি মোটরের জন্য খালি লিস্ট তৈরি
motor_targets = {f"Motor_{i}": [] for i in range(1, 6)}
valid_rows = []
base_sessions = []

df_raw = pd.read_csv(CSV_PATH)

for idx, row in df_raw.iterrows():
    filename = str(row['File Name'])
    match = re.match(FILENAME_PATTERN, filename)

    if match:
        state_digits = match.group(1) # ৫ ডিজিটের কোড এক্সট্র্যাক্ট করা (যেমন: '13201')

        for i in range(5):
            motor_targets[f"Motor_{i+1}"].append(int(state_digits[i]))

        base_session = filename.split('x')[0] # ডাটা লিক বন্ধ করতে বেস সেশন আইডি নেওয়া
        base_sessions.append(base_session)
        valid_rows.append(idx)

# পার্স করা লেবেল দিয়ে পরিচ্ছন্ন মেটাডাটা ফ্রেম তৈরি
metadata_df = df_raw.iloc[valid_rows].copy()
metadata_df['Base_Session_ID'] = base_sessions
for motor_name, labels in motor_targets.items():
    metadata_df[motor_name] = labels

print(f"Parsing complete! Valid audio files for processing: {len(metadata_df)}")

Parsing complete! Valid audio files for processing: 168


In [ ]:
# =============================================================
# Cell 3.5: Filter Distractor Classes (Pure Background) 46 samples
# Original cell 4
# =============================================================
# আপনার মূল ডেটাফ্রেমের নাম যদি 'df' হয়ে থাকে:
metadata_df = metadata_df[metadata_df['Description\n(Targets)'] != 'Absent (or disconnected)'].reset_index(drop=True)

print(f"Data filtered successfully! Remaining samples for feature extraction: {len(metadata_df)}")

Data filtered successfully! Remaining samples for feature extraction: 122


In [ ]:
# Cell 4: Audio Preprocessing and Feature Extraction (Mean + Std Version)
# Original cell 5
def filter_active_audio(y, top_db=20):
    """রেকর্ডিংয়ের শুরু/শেষ এবং মাঝখানের সম্পূর্ণ নীরব অংশ কেটে বাদ দেয়।"""
    intervals = librosa.effects.split(y, top_db=top_db)
    if len(intervals) == 0:
        return np.array([])
    return np.concatenate([y[start:end] for start, end in intervals])

def extract_combined_features(y_chunk, sr):
    """১০ সেকেন্ডের অ디오 চাঙ্ক থেকে Mean এবং Std বের করে ৩৪৮ সাইজের ভেক্টর বানায়।"""
    # ১. Mel-Spectrogram এক্সট্র্যাকশন
    mel = librosa.feature.melspectrogram(y=y_chunk, sr=sr, n_mels=N_MELS, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_mean = np.mean(mel_db, axis=1, keepdims=True)
    mel_std = np.std(mel_db, axis=1, keepdims=True) # স্ট্যান্ডার্ড ডেভিয়েশন

    # ২. MFCCs এক্সট্র্যাকশন
    mfcc = librosa.feature.mfcc(y=y_chunk, sr=sr, n_mfcc=N_MFCC, n_fft=N_FFT, hop_length=HOP_LENGTH)
    mfcc_mean = np.mean(mfcc, axis=1, keepdims=True)
    mfcc_std = np.std(mfcc, axis=1, keepdims=True) # স্ট্যান্ডার্ড ডেভিয়েশন

    # ৩. Spectral Contrast এক্সট্র্যাকশন
    contrast = librosa.feature.spectral_contrast(y=y_chunk, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_bands=N_BANDS)
    contrast_mean = np.mean(contrast[:N_BANDS], axis=1, keepdims=True)
    contrast_std = np.std(contrast[:N_BANDS], axis=1, keepdims=True) # স্ট্যান্ডার্ড ডেভিয়েশন

    # সব Mean এবং Std আলাদা আলাদা স্ট্যাক করা
    fused_mean = np.vstack([mel_mean, mfcc_mean, contrast_mean]) # (174, 1)
    fused_std = np.vstack([mel_std, mfcc_std, contrast_std])   # (174, 1)

    # চূড়ান্ত ভার্টিকাল স্ট্যাক (Mean + Std) -> (348, 1)
    fused = np.vstack([fused_mean, fused_std])
    return fused

In [ ]:
# Cell 5: Smart Feature Extraction with Auto-Save and Fast-Load Pipeline
# Original cell 6
import os
import numpy as np
import pandas as pd

# ১. ফিচার সেভ করার জন্য ড্রাইভের পাথ নির্ধারণ (Cell 2 এর AUDIO_DIR এর ভেতরেই সেভ হবে)
FEATURES_SAVE_PATH = os.path.join(os.path.dirname(AUDIO_DIR), "wolfset_348_features.npy")
METADATA_SAVE_PATH = os.path.join(os.path.dirname(AUDIO_DIR), "wolfset_segmented_metadata_348_features.csv")

# মাল্টি-টাস্ক ইমব্যালেন্স হ্যান্ডেল করার লজিক (ওভারল্যাপ ফাংশন)
def calculate_multitask_overlap(row):
    motor_states = [row[f"Motor_{i}"] for i in range(1, 6)]
    if any(state >= 3 for state in motor_states):
        return 0.90  # রেয়ার মোটর কন্ডিশন থাকলে ৯০% ওভারল্যাপ
    return 0.0

# ২. চেক করা হচ্ছে ড্রাইভে অলরেডি ফিচার ফাইল দুটি আছে কিনা
if os.path.exists(FEATURES_SAVE_PATH) and os.path.exists(METADATA_SAVE_PATH):
    print("="*60)
    print("PRE-EXTRACTED FEATURES FOUND! LOADING FROM GOOGLE DRIVE...")
    print("="*60)

    # ড্রাইভ থেকে সরাসরি লোড করা (সময় লাগবে মাত্র ২-৩ সেকেন্ড!)
    X_features = np.load(FEATURES_SAVE_PATH).astype(np.float32)
    final_meta_df = pd.read_csv(METADATA_SAVE_PATH)

    print(f"✓ Success: Loaded Feature Matrix Shape: {X_features.shape}")
    print(f"✓ Success: Loaded Segmented Metadata Rows: {len(final_meta_df)}")

else:
    print("="*60)
    print("NO PRE-SAVED FEATURES FOUND. STARTING AUDIO PROCESSING LOOP...")
    print("  (This will take time on the first run, but will save for future runs)")
    print("="*60)

    all_features = []
    segmented_metadata = []
    segment_idx = 1

    for idx, row in metadata_df.iterrows():
        filename = row['File Name']
        filepath = os.path.join(AUDIO_DIR, filename)

        if not os.path.exists(filepath):
            continue

        try:
            y, _ = librosa.load(filepath, sr=SR)
            y_active = filter_active_audio(y, top_db=20)

            if len(y_active) < SEGMENT_SAMPLES:
                continue

            overlap = calculate_multitask_overlap(row)
            step_samples = int(SEGMENT_SAMPLES * (1.5 - overlap))

            for start in range(0, len(y_active) - SEGMENT_SAMPLES + 1, step_samples):
                y_chunk = y_active[start:start + SEGMENT_SAMPLES]
                features = extract_combined_features(y_chunk, SR)

                all_features.append(features)

                record = {
                    "Segment_ID": f"SEG_{segment_idx:05d}",
                    "Source_File_ID": filename,
                    "Base_Session_ID": row['Base_Session_ID'],
                    "Description\n(Targets)": row['Description\n(Targets)']
                }
                for i in range(1, 6):
                    record[f"Motor_{i}"] = row[f"Motor_{i}"]
                segmented_metadata.append(record)
                segment_idx += 1

        except Exception as e:
            print(f"Error processing {filename}: {e}")

    # নম্পাই অ্যারে এবং পান্ডাস ডেটাফ্রেমে রূপান্তর
    X_features = np.array(all_features).astype(np.float32)
    final_meta_df = pd.DataFrame(segmented_metadata)

    # ৩. ভবিষ্যৎ রানগুলোর সময় বাঁচানোর জন্য ড্রাইভে পাকাপাকিভাবে সেভ করে রাখা
    print("\n Saving extracted features to Google Drive for future instant-loading...")
    np.save(FEATURES_SAVE_PATH, X_features)
    final_meta_df.to_csv(METADATA_SAVE_PATH, index=False)

    print(f"✓ Feature Extraction & Saving Complete! Shape: {X_features.shape}")

# --- BEGIN ROBUST COLUMN ENSURING AND FILTERING ---
# Step 1: Ensure 'Description\n(Targets)' column is present and up-to-date.
# This handles cases where final_meta_df is loaded from a pre-saved CSV that might not have this column,
# or has an outdated one. We will always re-merge from the filtered metadata_df.
if 'Description\n(Targets)' in final_meta_df.columns:
    final_meta_df = final_meta_df.drop(columns=['Description\n(Targets)'])

# Merge the description from the (already filtered) metadata_df
final_meta_df = final_meta_df.merge(
    metadata_df[['File Name', 'Description\n(Targets)']].rename(columns={'File Name': 'Source_File_ID'}),
    on='Source_File_ID',
    how='left'
)

# IMPORTANT: Strip whitespace from the target description column before filtering
final_meta_df['Description\n(Targets)'] = final_meta_df['Description\n(Targets)'].str.strip()

# Step 2: Filter out 'Absent (or disconnected)' samples and any NaN values from both final_meta_df and X_features.
initial_final_meta_df_len = len(final_meta_df)

# Get indices of rows that are NOT 'Absent (or disconnected)' AND are NOT NaN
valid_indices = final_meta_df[
    (final_meta_df['Description\n(Targets)'] != 'Absent (or disconnected)') &
    (final_meta_df['Description\n(Targets)'].notna())
].index

# Apply filtering to both final_meta_df and X_features
final_meta_df = final_meta_df.loc[valid_indices].reset_index(drop=True)
X_features = X_features[valid_indices] # Ensure X_features also gets filtered consistently

if len(final_meta_df) < initial_final_meta_df_len:
    print(f"  Note: Filtered {initial_final_meta_df_len - len(final_meta_df)} 'Absent (or disconnected)' (or NaN) samples from final_meta_df and X_features after merge.")
# --- END ROBUST COLUMN ENSURING AND FILTERING ---

# =====================================================================
# Cell 5: Smart Feature Extraction with Auto-Save and Fast-Load Pipeline
# Original cell 6
# UPDATED PART ONLY: paste this at the VERY END of your existing Cell 5
# =====================================================================

# -------------------- UPDATED: final alignment/sanity checks --------------------
# This protects against index mismatch after filtering Absent / NaN samples.

final_meta_df = final_meta_df.reset_index(drop=True)

assert len(final_meta_df) == len(X_features), (
    f"ERROR: final_meta_df rows ({len(final_meta_df)}) != X_features rows ({len(X_features)}). "
    "Feature-label alignment is broken."
)

required_cols = ["Base_Session_ID", "Description\n(Targets)"] + [f"Motor_{i}" for i in range(1, 6)]
missing_cols = [c for c in required_cols if c not in final_meta_df.columns]

assert len(missing_cols) == 0, f"ERROR: Missing columns in final_meta_df: {missing_cols}"

print("UPDATED Cell 5 check passed: X_features and final_meta_df are aligned.")
print("Final feature shape:", X_features.shape)
print("Final metadata rows:", len(final_meta_df))

PRE-EXTRACTED FEATURES FOUND! LOADING FROM GOOGLE DRIVE...
✓ Success: Loaded Feature Matrix Shape: (1665, 348, 1)
✓ Success: Loaded Segmented Metadata Rows: 1665
  Note: Filtered 192 'Absent (or disconnected)' (or NaN) samples from final_meta_df and X_features after merge.
UPDATED Cell 5 check passed: X_features and final_meta_df are aligned.
Final feature shape: (1473, 348, 1)
Final metadata rows: 1473


In [ ]:
# Cell 1: Polished Verification Function with Strict Type-Casting
# cell before final boom cell
# Run this after cell 5
import numpy as np
from collections import Counter

def verify_pipeline_precision(X_train_orig, X_train_bal, y_train_bal, X_val, y_val, motor_name, target_per_state):
    """
    এই ফাংশনটি আপনার ডেটা আর্কিটেকচারের জন্য একটি আলটিমেট সিকিউরিটি গার্ড।
    Dtype Mismatch বাগ ফিক্স সহ এটি এখন সম্পূর্ণ বুলেটপ্রুফ।
    """
    print(f"\n RUNNING RIGOROUS PRECISION TESTS FOR: {motor_name} ")
    print("-" * 65)

    # ---------------------------------------------------------
    # TEST 1: Exact Per-State Distribution Validation
    # ---------------------------------------------------------
    states, counts = np.unique(y_train_bal, return_counts=True)
    is_balanced = all(c == target_per_state for c in counts)

    if is_balanced:
        print(f" TEST 1 PASSED: Per-State Balancing is 100% Precise.")
    else:
        print(f" TEST 1 FAILED: Distribution Misalignment Detected!")
    for s, c in zip(states, counts):
        print(f"    State {s}: {c} samples (Expected: {target_per_state})")

    # ---------------------------------------------------------
    # TEST 2: Strict Data Leakage Check (Train vs Validation) WITH DTYPE FIX
    # ---------------------------------------------------------
    # float32 তে কাস্ট করা হচ্ছে যেন টাইপ ডিফারেন্সের কারণে লিকেজ মিস না হয়
    train_flat = X_train_bal.reshape(X_train_bal.shape[0], -1).astype(np.float32)
    val_flat = X_val.reshape(X_val.shape[0], -1).astype(np.float32)

    train_hashes = set(hash(row.tobytes()) for row in train_flat)
    val_hashes = set(hash(row.tobytes()) for row in val_flat)

    leakage_rows = train_hashes.intersection(val_hashes)

    if len(leakage_rows) == 0:
        print(f" TEST 2 PASSED: Zero Data Leakage! Validation set is 100% untouched.")
    else:
        print(f" TEST 2 CRITICAL FAILED: Data Leakage Detected! {len(leakage_rows)} rows overlapping with Val Set!")

    # ---------------------------------------------------------
    # TEST 3: Augmentation Integrity (Uniqueness Check)
    # ---------------------------------------------------------
    total_samples = len(X_train_bal)
    unique_samples = len(train_hashes)

    print(f" TEST 3 REPORT: Out of {total_samples} samples, {unique_samples} are unique feature patterns.")
    if unique_samples == total_samples:
        print("    Safe Jittering worked perfectly! No exact duplicate clones found.")
    else:
        print("    Note: Some exact clones present (Check if noise_strength is too low or 0).")

    print("-" * 65)

    if not is_balanced or len(leakage_rows) > 0:
        raise ValueError(f"🚨 Pipeline execution halted due to strict validation failure in {motor_name}!")

In [ ]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score

# ==========================================
# ১. আউটপুট ডিরেক্টরি এবং ট্র্যাকিং সেটআপ
# ==========================================
OUTPUT_DIR = "Thesis_Outputs_Individual_CNN"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ডাটা ট্র্যাকিং স্ট্রাকচার
history_tracking = {}
pooled_true_labels = {f"Motor_{i}": [] for i in range(1, 6)}
pooled_pred_labels = {f"Motor_{i}": [] for i in range(1, 6)}

# সেশন কলামের নাম ডাইনামিকলি খুঁজে বের করা
possible_session_cols = ['Session_ID', 'Session', 'session_id', 'session']
session_col = None
for col in possible_session_cols:
    if col in final_meta_df.columns:
        session_col = col
        break

if session_col is None:
    # যদি কোনো সেশন কলাম না পাওয়া যায়, তবে ইনডেক্স ব্যবহার করবে ব্যাকআপ হিসেবে
    print("⚠️ Warning: No explicit Session column found. Using index fallback.")
    final_meta_df['fallback_session'] = final_meta_df.index // 100
    session_col = 'fallback_session'

print(f"✅ Using '{session_col}' for validation session grouping.")

# ==========================================
# ২. ৫টি মোটরের জন্য মেইন ট্রেইনিং ও ইভ্যালুয়েশন লুপ
# ==========================================
for motor_idx in range(1, 6):
    motor_name = f"Motor_{motor_idx}"
    print("\n" + "="*70)
    print(f" EXECUTING PIPELINE FOR: {motor_name}")
    print("="*70)

    # [আপনার অরিজিনাল কোডের মডেল আর্কিটেকচার এবং ট্রেইনিং লুপ এখানে রান হবে]
    # (নোট: আপনার আগের কোডের ভেতরের 'groups = final_meta_df["Session_ID"]' লাইনটি
    # পরিবর্তন হয়ে নিচে অটোমেটিক 'groups = final_meta_df[session_col]' হিসেবে কাজ করবে)

    # উদাহরণস্বরূপ লুপের ভেতরে সেশন গ্রুপ করার লাইনটি এভাবে আপডেট হবে:
    # groups = final_meta_df[session_col].values

    # -------------------------------------------------------------
    # ডাটা কালেকশন পার্ট (যা লুপের একদম শেষে ইভ্যালুয়েশনের সময় কাজ করবে):
    # -------------------------------------------------------------
    # (ধরে নিচ্ছি y_true_val এবং y_pred_val হলো আপনার ভ্যালিডেশন সেটের আসল ও প্রেডিক্টেড লেবেল)
    # pooled_true_labels[motor_name].extend(y_true_val)
    # pooled_pred_labels[motor_name].extend(y_pred_val)

⚠️ Warning: No explicit Session column found. Using index fallback.
✅ Using 'fallback_session' for validation session grouping.

 EXECUTING PIPELINE FOR: Motor_1

 EXECUTING PIPELINE FOR: Motor_2

 EXECUTING PIPELINE FOR: Motor_3

 EXECUTING PIPELINE FOR: Motor_4

 EXECUTING PIPELINE FOR: Motor_5


In [ ]:

# =====================================================================
# FINAL CELL: Dedicated CNN Architecture, K-Fold Cross-Validation,
# Physics-Informed Augmentation, and Thesis-Grade Performance Representation
# =====================================================================

import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, BatchNormalization, ReLU, MaxPooling1D, Flatten, Dense, Dropout, Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

# --- আউটপুট ডিরেক্টরি সেটআপ ---
OUTPUT_DIR = "Thesis_Outputs_Individual_CNN"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- ১. থিসিস গ্রেড ডাটা ট্র্যাকিং স্ট্রাকচার ---
history_tracking = {}
pooled_true_labels = {f"Motor_{i}": [] for i in range(1, 6)}
pooled_pred_labels = {f"Motor_{i}": [] for i in range(1, 6)}

# --- Dedicated CNN Architecture ---
def build_dedicated_cnn(input_shape, num_classes):
    model = Sequential([
        Input(shape=input_shape), # Added Input layer to remove UserWarning
        Conv1D(filters=16, kernel_size=3, strides=1, padding='same'),
        BatchNormalization(),
        ReLU(),
        MaxPooling1D(pool_size=2),

        Conv1D(filters=32, kernel_size=3, strides=1, padding='same'),
        BatchNormalization(),
        ReLU(),
        MaxPooling1D(pool_size=2),

        Flatten(),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(num_classes, activation='softmax')
    ])
    return model

# --- ৫টি মোটরের জন্য ক্রস-ভ্যালিডেশন এবং ট্রেইনিং লুপ ---
trained_models_per_motor = {f"Motor_{i}": [] for i in range(1, 6)}

for i in range(1, 6):
    motor_key = f"Motor_{i}"
    print("\n" + "="*70)
    print(f" EXECUTING PIPELINE FOR: {motor_key}")
    print("="*70)

    # আপনার কোডের অরিজিনাল ডেটা ফিল্টারিং ও এক্সট্রাকশন অংশ
    motor_indices = final_meta_df[final_meta_df[motor_key].notna()].index
    X_motor = X_features[motor_indices]
    y_motor = final_meta_df.loc[motor_indices, motor_key].values.astype(int)
    groups_motor = final_meta_df.loc[motor_indices, 'Base_Session_ID'].values

    # লেবেল এনকোড করা
    le = LabelEncoder()
    y_motor_encoded = le.fit_transform(y_motor)
    num_classes = len(le.classes_)

    # ৫-ফোল্ড গ্রুপ কে-ফোল্ড ক্রস ভ্যালিডেশন
    gkf = GroupKFold(n_splits=5)

    # এই মোটরের অল-ফোল্ড হিস্টোরি জমানোর লিস্ট
    history_tracking[motor_key] = []

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_motor, y_motor_encoded, groups=groups_motor)):
        X_train_split, X_val_split = X_motor[train_idx], X_motor[val_idx]
        y_train_split, y_val_split = y_motor_encoded[train_idx], y_motor_encoded[val_idx]

        # ট্রেইনিং সেটে ক্লাস ব্যালেন্সিং ও ফিজিক্স অগমেন্টেশন
        unique_states = np.unique(y_train_split)
        X_train_balanced_list = []
        y_train_balanced_list = []

        target_samples_per_state = 120  # প্রতিটি স্টেটের জন্য টার্গেট স্যাম্পল

        for state in unique_states:
            state_indices = np.where(y_train_split == state)[0]
            X_state = X_train_split[state_indices]

            if len(state_indices) >= target_samples_per_state:
                # ডাউনস্যাম্পলিং
                sampled_indices = np.random.choice(state_indices, target_samples_per_state, replace=False)
                X_train_balanced_list.append(X_train_split[sampled_indices])
                y_train_balanced_list.append(np.full(target_samples_per_state, state))
            else:
                # ফিজিক্স-ইনফর্মড অ্যাকোস্টিক অগমেন্টেশন (ওভারস্যাম্পলিং)
                X_train_balanced_list.append(X_state)
                y_train_balanced_list.append(np.full(len(state_indices), state))

                needed = target_samples_per_state - len(state_indices)
                # ফেজ শিফট ও নয়েজ ইঞ্জেকশন দিয়ে অগমেন্টেশন
                augmented_chunks = []
                for _ in range(needed):
                    base_pattern = X_state[np.random.choice(len(X_state))]
                    # Phase Shift Simulation
                    shifted_pattern = np.roll(base_pattern, shift=np.random.randint(-5, 5), axis=0)
                    # Micro-Acoustic Thermal Noise Simulation
                    noise = np.random.normal(0, 0.005, size=shifted_pattern.shape)
                    augmented_chunks.append(shifted_pattern + noise)

                X_train_balanced_list.append(np.array(augmented_chunks))
                y_train_balanced_list.append(np.full(needed, state))

        X_train_balanced = np.vstack(X_train_balanced_list)
        y_train_balanced = np.concatenate(y_train_balanced_list)

        # স্কেলিং
        scaler = StandardScaler()
        X_train_shape = X_train_balanced.shape
        X_train_flat = X_train_balanced.reshape(X_train_shape[0], -1)
        X_train_scaled_flat = scaler.fit_transform(X_train_flat)
        X_train_balanced = X_train_scaled_flat.reshape(X_train_shape)

        X_val_shape = X_val_split.shape
        X_val_flat = X_val_split.reshape(X_val_shape[0], -1)
        X_val_scaled_flat = scaler.transform(X_val_flat)
        X_val_scaled = X_val_scaled_flat.reshape(X_val_shape)

        # ইনপুট শেপ ও মডেল কম্পাইলেশন
        input_shape = (X_train_balanced.shape[1], X_train_balanced.shape[2])
        model = build_dedicated_cnn(input_shape, num_classes)
        model.compile(
            optimizer=Adam(learning_rate=0.001),
            loss='sparse_categorical_crossentropy',
            metrics=['accuracy']
        )

        callbacks = [tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=12, restore_best_weights=True)]

        # মডেল ফিট এবং হিস্টোরি ট্র্যাকিং
        history = model.fit(
            X_train_balanced, y_train_balanced,
            validation_data=(X_val_scaled, y_val_split),
            epochs=80,
            batch_size=16,
            callbacks=callbacks,
            verbose=0
        )

        # হিস্টোরি অবজেক্টের ডিকশনারি ডাটা সেভ করে রাখা
        history_tracking[motor_key].append(history.history)
        trained_models_per_motor[motor_key].append(model)
        print(f" -> {motor_key} Fold {fold+1} Training Complete.")

        # প্রেডিকশন এবং লেবেল পুলিং (ভ্যালিডেশন সেটের আসল ক্লাস নাম রিট্রিভ করে)
        preds = np.argmax(model.predict(X_val_scaled, verbose=0), axis=1)

        # এনকোড করা লেবেলকে আসল লেবেলে ফেরত নেওয়া (যেমন: ০, ১, ২, ৩, ৪)
        true_original_labels = le.inverse_transform(y_val_split)
        pred_original_labels = le.inverse_transform(preds)

        pooled_true_labels[motor_key].extend(true_original_labels)
        pooled_pred_labels[motor_key].extend(pred_original_labels)

print("\n🚀 [SUCCESS] অল-মোটর ট্রেইনিং এবং ডাটা কালেকশন সম্পূর্ণ হয়েছে।")

# =====================================================================
# 📊 ২. থিসিস রিপ্রেজেন্টেশন জেনারেটর (গ্রাফ এবং সিএসভি অটো-সেভ)
# =====================================================================

print("\n📊 জেনারেটিং থিসিস-গ্রেড পারফরম্যান্স মেট্রিক্স চার্ট...")
all_states = [0, 1, 2, 3, 4]
f1_records = []

for motor_key in pooled_true_labels.keys():
    # সম্পূর্ণ ওওভি-সেভড ও পুলড ডেটার ওপরে ক্লাসিফিকেশন রিপোর্ট তৈরি
    rep = classification_report(
        pooled_true_labels[motor_key],
        pooled_pred_labels[motor_key],
        output_dict=True,
        zero_division=0
    )

    motor_f1s = {"Motor": motor_key}
    for state in all_states:
        state_str = str(state)
        # যদি কোনো স্টেট টেস্ট সেটে একদমই না থাকে (Support 0), থিসিসে তা 0 দেখাবে
        motor_f1s[f"State_{state}"] = rep[state_str]['f1-score'] if state_str in rep else 0.0

    # ম্যাক্রো এবং ওয়েটেড এভারেজ যোগ করা
    motor_f1s["Macro_Avg"] = rep['macro avg']['f1-score']
    f1_records.append(motor_f1s)

# ডাটাফ্রেম তৈরি ও সিএসভি সেভ
df_f1 = pd.DataFrame(f1_records)
csv_path = os.path.join(OUTPUT_DIR, "individual_cnn_f1_scores.csv")
df_f1.to_csv(csv_path, index=False)
print(f"✅ F1-Scores CSV সফলভাবে সেভ হয়েছে এখানে: {csv_path}")

# --- সুন্দর এবং স্পষ্ট গ্রুপড বার চার্ট তৈরি ---
plt.figure(figsize=(12, 6))
x = np.arange(len(df_f1["Motor"]))
width = 0.15

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

for idx, state in enumerate(all_states):
    plt.bar(x + (idx - 2) * width, df_f1[f"State_{state}"], width, label=f"State {state}", color=colors[idx])

plt.xlabel("Fault Diagnosis Target (Motors)", fontsize=12, fontweight='bold')
plt.ylabel("Macro F1-Score", fontsize=12, fontweight='bold')
plt.title("Individual 1D-CNN Fleet Performance across Fault States (Support > 0)", fontsize=14, fontweight='bold', pad=15)
plt.xticks(x, df_f1["Motor"], fontsize=11)
plt.ylim(0, 1.05)
plt.legend(title="Motor States", loc='lower left')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()

graph_path = os.path.join(OUTPUT_DIR, "individual_cnn_f1_representation.png")
plt.savefig(graph_path, dpi=300)
plt.show()
print(f"✅ থিসিস গ্রাফ সফলভাবে সেভ হয়েছে এখানে: {graph_path}")

# =====================================================================
# 📋 ৩. ফিল্টারড থিসিস রিপোর্ট জেনারেটর (Support > 0 ফিল্টারসহ)
# =====================================================================

report_txt_path = os.path.join(OUTPUT_DIR, "pooled_classification_report.txt")

with open(report_txt_path, "w", encoding="utf-8") as f:
    f.write("==================================================\n")
    f.write(" POOLED CLASSIFICATION REPORT (INDIVIDUAL CNN)   \n")
    f.write("==================================================\n\n")

    for motor_key in pooled_true_labels.keys():
        f.write(f"\n###########################\n {motor_key} FINAL RESULTS \n###########################\n")

        # ইউনিক লেবেল বের করা যেগুলো বাস্তবে টেস্ট সেটে উপস্থিত ছিল (Support > 0)
        unique_true_labels = sorted(list(set(pooled_true_labels[motor_key])))

        report_text = classification_report(
            pooled_true_labels[motor_key],
            pooled_pred_labels[motor_key],
            labels=unique_true_labels,
            zero_division=0
        )
        f.write(report_text)
        print(f"\n➔ {motor_key} এর ফিল্টারড থিসিস রিপোর্ট প্রস্তুত।")

print(f"\n📋 চূড়ান্ত টেক্সট রিপোর্ট সফলভাবে সেভ হয়েছে এখানে: {report_txt_path}")


 EXECUTING PIPELINE FOR: Motor_1
 -> Motor_1 Fold 1 Training Complete.
 -> Motor_1 Fold 2 Training Complete.
 -> Motor_1 Fold 3 Training Complete.
 -> Motor_1 Fold 4 Training Complete.
 -> Motor_1 Fold 5 Training Complete.

 EXECUTING PIPELINE FOR: Motor_2
 -> Motor_2 Fold 1 Training Complete.
 -> Motor_2 Fold 2 Training Complete.
 -> Motor_2 Fold 3 Training Complete.
 -> Motor_2 Fold 4 Training Complete.
 -> Motor_2 Fold 5 Training Complete.

 EXECUTING PIPELINE FOR: Motor_3
 -> Motor_3 Fold 1 Training Complete.
 -> Motor_3 Fold 2 Training Complete.
 -> Motor_3 Fold 3 Training Complete.
 -> Motor_3 Fold 4 Training Complete.
 -> Motor_3 Fold 5 Training Complete.

 EXECUTING PIPELINE FOR: Motor_4
 -> Motor_4 Fold 1 Training Complete.
 -> Motor_4 Fold 2 Training Complete.
 -> Motor_4 Fold 3 Training Complete.
 -> Motor_4 Fold 4 Training Complete.
 -> Motor_4 Fold 5 Training Complete.

 EXECUTING PIPELINE FOR: Motor_5
 -> Motor_5 Fold 1 Training Complete.
 -> Motor_5 Fold 2 Training Comp